# 3.19 NIR Heater Noise

OVERVIEW: Observe a relatively empty position on the sky while adjusting heater setpoint and deadband. Measure the 


DATA PRODUCTS:
- table of heater settings and thermal background statistics


EXIT CRITERIA: observation obtained, analysis complete and heater #4 settings adjusted


DATA VOLUME: We plan for XX observations with the NIR science subarray.
- one NIR region 80x400


TARGET CRITERIA: Select one region with minimal NIR sources in the NIRDA science FOV.


KEY STAKEHOLDERS: Trevor Foote

## Load In

In [ ]:
''' Make sure you have pandorasim installed and updated '''
import pandorasat as ps
import pandorasim as pp
ps.utils.get_phoenix_model(teff=7000, jmag=10) # this is a temporary fix to an import error with pandorasim
from pandorasim import VisibleSim, NIRSim
from pandorasim import plot_nirda_integrations
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord, Longitude
from astropy.io import fits
from pandorasat.plotting import animate
from pandorasat.plotting import save_mp4, save_gif
from astropy.time import Time, TimeDelta

### Importing additional functions and constants used in this notebook
import sys, os
sys.path.append(os.path.abspath('..'))
from CommissFunctions import generate_task_plan 
from CommissFunctions import data_rate, bits_per_pix_VIS, compression_fractor_VIS, frame_time_VIS, stored_frames_per_int_VIS, pass_time_min, VIS_ra_shape, VIS_dec_shape, NIR_ra_shape, NIR_dec_shape, regions_NIR, bits_per_pix_NIR, compression_fractor_NIR 


In [ ]:
''' Define a PandoraSat object for calling constants '''
p = ps.PandoraSat()

## Define Target

In [ ]:
### Possible Targets:
# empty_region = SkyCoord(174.2471975, -16.8307524, unit='deg')

In [ ]:
''' Finds target coordinates based on astropy '''
### You can do this differently as long as you have DEC and RA
target = "pnt131.55300+010_49200" # empty position on sky
empty_region = SkyCoord(131.55300, 10.49200, unit='deg') # coordinates for "target" from GAIA database with more recent epoch then SkyCoord pulls from name

## Define Heater parameter space

In [ ]:
import itertools

### IF YOU CHANGE SETPOINTS OR DEADBANDS, YOU MUST USE THE Thermistor-Heater Worksheet ON LLNL TEAMS TO DETERMINE NEW ADC VALUES
####################################
set_points = [-30, -25, -20]
set_points_adc = [3904, 3834, 3745]
deadbands = [0.5, 1, 5, 10]
deadbands_adc = [6, 13, 70, 159]
####################################


heater_settings = np.array(list(itertools.product(set_points, deadbands)))
heater_settings_adc = np.array(list(itertools.product(set_points_adc, deadbands_adc)))

### Set the number of observations here
num_observations = len(heater_settings)

In [ ]:
num_observations

## Data Volume

12 observations and we only need the NIR data to downlink

### VIS Observations

In [ ]:
##### VIS Observation #####
regions_VIS           = 1 #2
VIS_xpix              = 100
VIS_ypix              = 100
frames_per_int_VIS     = 1
num_int_VIS            = 3000
int_and_reset_time_VIS = frame_time_VIS * frames_per_int_VIS
bits_per_int_VIS      = VIS_xpix * VIS_ypix * regions_VIS * bits_per_pix_VIS * stored_frames_per_int_VIS
bits_per_sec_VIS       = bits_per_int_VIS / int_and_reset_time_VIS
test_time_VIS          = num_int_VIS * int_and_reset_time_VIS
bits_per_sec_comp_VIS  = bits_per_sec_VIS / compression_fractor_VIS
bits_test_total_VIS    = bits_per_sec_comp_VIS * test_time_VIS
Gbits_test_packet_VIS  = bits_test_total_VIS / 1E9 * 1.1 * 1.25
downlinks_VIS          = Gbits_test_packet_VIS * 1E9 / (data_rate * 1E6) /60 /pass_time_min 

print('VIS Observations:', Gbits_test_packet_VIS*num_observations, 'Gbits')
print('Test time:', test_time_VIS)
print('Downlinks:', downlinks_VIS*num_observations)

### NIR Observations

In [ ]:
##### NIR Observation - Science window ##### 
NIR_xpix_strt         = 1968 
NIR_ypix_strt         = 824
NIR_xpix              = 80
NIR_ypix              = 400
num_int_NIR           = 65
stored_frames_per_int_NIR = 24
frames_per_int_NIR        = 24 
frame_time_NIR            = (NIR_xpix+12) * (NIR_ypix+2) * 0.00001 #sec
int_and_reset_time_NIR    = frame_time_NIR * (frames_per_int_NIR+1)
bits_per_int_NIR          = NIR_xpix * NIR_ypix * regions_NIR * bits_per_pix_NIR * stored_frames_per_int_NIR
bits_per_sec_NIR          = bits_per_int_NIR / int_and_reset_time_NIR
test_time_NIR             = num_int_NIR * int_and_reset_time_NIR
bits_per_sec_comp_NIR     = bits_per_sec_NIR / compression_fractor_NIR
bits_test_total_NIR       = bits_per_sec_comp_NIR * test_time_NIR
Gbits_test_packet_NIR     = bits_test_total_NIR / 1E9 * 1.1 * 1.25
downlinks_NIR             = Gbits_test_packet_NIR * 1E9 / (data_rate * 1E6) /60 /pass_time_min

print('NIR Observations:', Gbits_test_packet_NIR * num_observations, 'Gbits')
print('Test time:', test_time_NIR)
print('Downlinks:', downlinks_NIR*num_observations)

### Total Data Volume
#### assuming we have to downlink VISDA data

In [ ]:
### Data volume
Gbits_total = num_observations*(Gbits_test_packet_VIS+Gbits_test_packet_NIR)
downlinks_total = num_observations*(downlinks_VIS+downlinks_NIR)
test_time_total = num_observations*max([test_time_VIS, test_time_NIR])

print('Test time, total, no slew/overhead time:', test_time_total/60/60, 'hours')
print('Volume, total:', Gbits_total, 'Gbits')
print('Downlinks, total:', downlinks_total)

In [ ]:
### Adding important variables to a dictionary for later
obs_dict = {
    'VIS': {'test_time': test_time_VIS, 'xpix': VIS_xpix, 'ypix': VIS_ypix, 'frames_per_int': frames_per_int_VIS, 'num_int': num_int_VIS},
    'NIR': {'test_time': test_time_NIR, 'xpix_start': NIR_xpix_strt, 'xpix': NIR_xpix, 'ypix_start': NIR_ypix_strt, 'ypix': NIR_ypix, 
             'frames_per_int': frames_per_int_NIR, 'num_int': num_int_NIR},

}

obs_dict["VIS"], obs_dict["NIR"]

In [ ]:
''' Identify the start time and end time of the observation '''
start_time = Time.now() # starting now. This should get overridden by the scheduler. 
test_time = max(obs_dict['VIS']['test_time'], obs_dict['NIR']['test_time'])
time_delta = TimeDelta(test_time, format='sec')
end_time = start_time + time_delta

## Generate SOC File

### To run all 12 observations together

In [ ]:
for i in range(num_observations):
    observation_number=i
    heater_setting_adc = heater_settings_adc[i]

    variables = {
        'visit_id': '0319', ### setting based on task number. Could change. 
        'obs_id': f"{observation_number:03}", ### setting based on observation number. 
        'target': target, 
        'priority': 1, ### all commissioning is priority 1
        'start_time': start_time, 
        'stop_time': end_time, 
        'RA': empty_region.ra.deg,
        'DEC': empty_region.dec.deg, 
        
        'NIR_AvgGroups': 0, 
        'NIR_ROI_StartX': obs_dict['NIR']['xpix_start'], 
        'NIR_ROI_StartY': obs_dict['NIR']['ypix_start'],
        'NIR_ROI_SizeX': obs_dict['NIR']['xpix'], 
        'NIR_ROI_SizeY': obs_dict['NIR']['ypix'],
        'NIR_SC_Resets1': 1,
        'NIR_SC_Resets2': 1, 
        'NIR_SC_DropFrames1': 0,
        'NIR_SC_DropFrames2': 0, 
        'NIR_SC_DropFrames3': 0, 
        'NIR_SC_ReadFrames': obs_dict['NIR']['frames_per_int'],
        'NIR_targetID': target, 
        'NIR_SC_Groups': 1, 
        'NIR_SC_Integrations': obs_dict['NIR']['num_int'], 
        
        'ffi_flag': 0, # this task does not require full frame images
        'VIS_StarRoiDetMethod': 1,  ### 0: monitor stars at positions defined in [PredefinedStarRoiRa, PredefinedStarRoiDec]. 1: Run star detection algorithm on first image and obtain star ROIs from results.
        'VIS_FramesPerCoadd': obs_dict['VIS']['frames_per_int'], 
        'VIS_NumTotalFramesRequested': obs_dict['VIS']['num_int'] * obs_dict['VIS']['frames_per_int'], 
        'VIS_TargetRA': empty_region.ra.deg,
        'VIS_TargetDEC': empty_region.dec.deg, 
        'VIS_IncludeFieldSolnsInResp': 1, ### Determines whether the Ra/Dec/Rot values calculated for each frame in the visible science collect will be included in the response message [1] or not [0].
        'VIS_StarRoiDimension': [obs_dict['VIS']['xpix'], obs_dict['VIS']['ypix']], 
        'VIS_MaxNumStarRois': regions_VIS, ### max number of star ROIs that will be used for coadding signal from target stars if StarRoiDetMethod == 1.
        'VIS_numPredefinedStarRois': regions_VIS, ### If StarRoiDetMethod==0, this is the number of [RA, DEC] coordinates contained in the PredefinedStarRoiRA and PredefinedStarRoiDEC vectors.
        'VIS_PredefinedStarRoiRa': [empty_region.ra.deg], 
        'VIS_PredefinedStarRoiDec': [empty_region.dec.deg], 
        'VIS_targetID': target, 
        'VIS_NumExposuresMax': obs_dict['VIS']['num_int'] * obs_dict['VIS']['frames_per_int'], 
        'VIS_ExposureTime_us': int(frame_time_VIS * 1000000), 

        'heater_4': True,
        'H4_SetPoint': heater_setting_adc[0],
        'H4_Deadband': heater_setting_adc[1]

    }

    '''Generate the SOC .xml file'''

    output_soc_file = variables['visit_id']+'_'+variables['obs_id']+'_SOC.xml' 
    generate_task_plan(variables, output_soc_file)